# Padel Stroke Classification from Raw Video (I3D · SlowFast · Transformers)

This notebook benchmarks three families of video models on the padel stroke dataset: (1) **I3D** (inflated 3D CNN), (2) **SlowFast** two-pathway networks, and (3) **Video Transformers** (VideoMAE / TimeSformer). Each section loads pre-trained weights, adapts the classifier head to the padel labels, and runs a short fine-tuning cycle suitable for limited data.

## Roadmap
1. **Install deps** – PyTorchVideo, TorchVision ≥0.15, HuggingFace Transformers, Decord.
2. **Inspect dataset** – derive manifest of `dataset/<stroke>/*.mp4`.
3. **Clip sampler** – uniform temporal sampling with on-the-fly augmentation.
4. **Common dataloaders** – PyTorch dataset that outputs tensors for different architectures.
5. **I3D fine-tuning** – load `i3d_r50` pre-trained on Kinetics-400.
6. **SlowFast fine-tuning** – load `slowfast_r50` pre-trained on Kinetics-600.
7. **Video Transformer (VideoMAE-Small)** – start from MAE++ weights and parameter-efficient fine-tune.
8. **Evaluation + export** – shared validation metrics, confusion matrix, and experiment tracking.

In [ ]:
# Optional one-time dependency installation (comment out if already satisfied)
import sys
if 'google.colab' in sys.modules:
    !pip install -q torch torchvision torchaudio pytorchvideo decord einops timm transformers accelerate
else:
    print('✓ Using existing local environment (ensure pytorchvideo, decord, transformers are installed).')

In [ ]:
from __future__ import annotations

import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision.transforms import Compose
from torchvision.transforms import functional as F
from torchvision.transforms import InterpolationMode

from pytorchvideo.transforms import UniformTemporalSubsample
from pytorchvideo.data.encoded_video import EncodedVideo
from pytorchvideo.models.hub import i3d_r50, slowfast_r50

from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification, AutoConfig

DATASET_ROOT = Path('../dataset').resolve()
ARTIFACTS = Path('../artifacts/raw_video')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 1. Dataset manifest
Create a dataframe with columns: `path`, `stroke`, `duration`, `frames`, and `split`.

In [ ]:
import cv2

def build_manifest(data_root: Path) -> pd.DataFrame:
    rows = []
    for stroke_dir in sorted(data_root.iterdir()):
        if not stroke_dir.is_dir():
            continue
        for video_path in sorted(stroke_dir.glob('*.mp4')):
            cap = cv2.VideoCapture(str(video_path))
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
            duration = frames / fps if fps else 0
            cap.release()
            rows.append({
                'path': str(video_path.resolve()),
                'stroke': stroke_dir.name,
                'frames': frames,
                'fps': fps,
                'duration': duration
            })
    return pd.DataFrame(rows)

manifest = build_manifest(DATASET_ROOT)
manifest.groupby('stroke').size().sort_values(ascending=False)

## 2. Train/val/test split
Use a stratified split (70/20/10) with deterministic seed.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(manifest, test_size=0.3, stratify=manifest['stroke'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.3333, stratify=temp_df['stroke'], random_state=SEED)
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'
manifest = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
manifest.to_csv(ARTIFACTS / 'manifest.csv', index=False)
manifest.head()

## 3. Video dataset abstraction
A single dataset class that can emit either a single-stream tensor (I3D) or dual-stream (SlowFast) and also supply frame stacks for transformers.

In [ ]:
class PadelVideoDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, split: str, frames_per_clip: int = 64, resize: int = 256, crop_size: int = 224):
        self.df = dataframe[dataframe['split'] == split].reset_index(drop=True)
        self.split = split
        self.frames_per_clip = frames_per_clip
        self.resize = resize
        self.crop_size = crop_size
        self.labels = sorted(dataframe['stroke'].unique())
        self.label2id = {label: idx for idx, label in enumerate(self.labels)}
        self.temporal_sampler = UniformTemporalSubsample(frames_per_clip)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video = EncodedVideo.from_path(row['path'])
        duration = video.duration
        clip = video.get_clip(start_sec=0.0, end_sec=duration)
        video_data = clip['video']  # (C, T, H, W) float32 in [0,1]
        video_data = self.temporal_sampler(video_data)
        video_data = F.resize(video_data, [self.resize], interpolation=InterpolationMode.BILINEAR)
        if self.split == 'train':
            video_data = F.center_crop(video_data, [self.crop_size, self.crop_size])
        else:
            video_data = F.center_crop(video_data, [self.crop_size, self.crop_size])
        video_data = (video_data - 0.5) / 0.5  # normalize
        label = self.label2id[row['stroke']]
        return video_data, label

train_dataset = PadelVideoDataset(manifest, 'train')
val_dataset = PadelVideoDataset(manifest, 'val')
test_dataset = PadelVideoDataset(manifest, 'test')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

## 4. Helper utilities
Shared training loop, evaluation metrics, and experiment logging.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from torch.optim import AdamW

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses, preds, labels = [], [], []
    device = next(model.parameters()).device
    for batch in loader:
        frames, target = batch
        frames = frames.to(device)
        target = torch.tensor(target, device=device) if not torch.is_tensor(target) else target.to(device)
        logits = model(frames)
        loss = criterion(logits, target)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        losses.append(loss.item())
        preds.append(logits.argmax(dim=1).detach().cpu())
        labels.append(target.detach().cpu())
    preds = torch.cat(preds)
    labels = torch.cat(labels)
    return np.mean(losses), accuracy_score(labels, preds)

def fit_model(model, train_loader, val_loader, epochs=10, lr=1e-4):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.05)
    best_state, best_acc = None, 0.0
    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        print(f"Epoch {epoch+1:02d} | train: loss {train_loss:.3f}, acc {train_acc:.2%} | val: loss {val_loss:.3f}, acc {val_acc:.2%}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ARTIFACTS / f"{model.__class__.__name__}_best.pth")
    return model, best_acc

## 5. I3D baseline
We start from `i3d_r50` pre-trained on Kinetics-400 using RGB modality. Only the classification head is randomly initialized.

In [ ]:
def build_i3d(num_classes: int):
    model = i3d_r50(pretrained=True)
    model.blocks[-1].proj = nn.Linear(model.blocks[-1].proj.in_features, num_classes)
    return model

i3d_model = build_i3d(num_classes=len(train_dataset.labels)).to('cuda' if torch.cuda.is_available() else 'cpu')
i3d_model, i3d_best_acc = fit_model(i3d_model, train_loader, val_loader, epochs=15, lr=5e-5)

## 6. SlowFast baseline
SlowFast expects a tuple of slow and fast pathways with different temporal resolutions. We adapt `PadelVideoDataset` output for this section.

In [ ]:
def make_slowfast_inputs(frames: torch.Tensor, alpha: int = 4):
    # frames shape: (B, C, T, H, W)
    fast_pathway = frames
    slow_pathway = frames[:, :, ::alpha, :, :]
    return [slow_pathway, fast_pathway]

class SlowFastWrapper(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.backbone.blocks[-1].proj = nn.Linear(self.backbone.blocks[-1].proj.in_features, num_classes)
    def forward(self, x):
        slowfast_x = make_slowfast_inputs(x)
        return self.backbone(slowfast_x)

slowfast_backbone = slowfast_r50(pretrained=True)
slowfast_model = SlowFastWrapper(slowfast_backbone, len(train_dataset.labels)).to('cuda' if torch.cuda.is_available() else 'cpu')
slowfast_model, slowfast_best_acc = fit_model(slowfast_model, train_loader, val_loader, epochs=15, lr=5e-5)

## 7. Video Transformer baseline
We fine-tune a pre-trained VideoMAE base model with optional LoRA adapters to keep the number of trainable parameters small.

In [ ]:
peft_available = False
try:
    from peft import LoraConfig, get_peft_model
    peft_available = True
except ImportError:
    print('Install peft for LoRA fine-tuning (pip install peft). Falling back to full fine-tune.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = VideoMAEImageProcessor.from_pretrained('MCG-NJU/videomae-base')
videomae_core = VideoMAEForVideoClassification.from_pretrained(
    'MCG-NJU/videomae-base',
    label2id=train_dataset.label2id,
    id2label={v: k for k, v in train_dataset.label2id.items()}
)
if peft_available:
    lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['query','value'])
    videomae_core = get_peft_model(videomae_core, lora_config)
videomae_core.to(device)

class VideoMAEWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
    def forward(self, pixel_values):
        return self.base_model(pixel_values=pixel_values).logits

videomae_model = VideoMAEWrapper(videomae_core).to(device)

def collate_videomae(batch):
    videos, labels = zip(*batch)
    inputs = processor(list(videos), return_tensors='pt')
    return inputs['pixel_values'], torch.tensor(labels)

train_loader_vm = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_videomae)
val_loader_vm = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_videomae)
videomae_model, videomae_best_acc = fit_model(videomae_model, train_loader_vm, val_loader_vm, epochs=8, lr=1e-4)

## 8. Evaluation
Use the `test_dataset` split for final metrics and confusion matrices.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, loader, labels):
    device = next(model.parameters()).device
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for frames, target in loader:
            frames = frames.to(device)
            logits = model(frames)
            preds.append(logits.argmax(dim=1).cpu())
            gts.append(target)
    preds = torch.cat(preds)
    gts = torch.cat(gts)
    print(classification_report(gts, preds, target_names=labels))
    cm = confusion_matrix(gts, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
evaluate_model(i3d_model, test_loader, train_dataset.labels)
evaluate_model(slowfast_model, test_loader, train_dataset.labels)
test_loader_vm = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_videomae)
evaluate_model(videomae_model, test_loader_vm, train_dataset.labels)

## 9. Experiment tracking & export
Log each experiment (model, hyperparameters, metrics) to a JSONL file for later comparison with the skeleton-based pipeline.

In [ ]:
import datetime as dt

def log_run(model_name, metrics: Dict):
    payload = {'model': model_name, 'timestamp': dt.datetime.utcnow().isoformat(), **metrics}
    with open(ARTIFACTS / 'experiments.jsonl', 'a') as f:
        f.write(json.dumps(payload) + '\n')

log_run('I3D', {'val_acc': float(i3d_best_acc), 'notes': '15 epochs, lr=5e-5'})
log_run('SlowFast', {'val_acc': float(slowfast_best_acc), 'notes': '15 epochs, lr=5e-5'})
log_run('VideoMAE', {'val_acc': float(videomae_best_acc), 'notes': '8 epochs, lr=1e-4'})